# LUNA16 subset0 — Download & Save as Kaggle Dataset

**Before running:**
- Make sure internet is **enabled** in Notebook settings (right panel → Internet → On)
- Accelerator can be **None** — this is just download + unzip, no GPU needed
- This notebook's `/kaggle/working` has 20GB free — subset0 unzipped is ~12GB, fits fine

**After running:**
- Go to the notebook's output panel → click **'Save Version'** → tick **'Save output files'**
- Then go to the output tab and click **'Create dataset from output'**
- Name it something like `luna16-subset0-raw-ct`
- Then add that dataset to your preprocessing notebook

In [1]:
import os

WORKING_DIR = '/kaggle/working'
ZIP_PATH    = os.path.join(WORKING_DIR, 'subset0.zip')
EXTRACT_DIR = os.path.join(WORKING_DIR, 'luna16_subset0')

# Zenodo direct download URL for LUNA16 subset0
# Source: https://zenodo.org/records/3723295
ZENODO_URL = 'https://zenodo.org/records/3723295/files/subset0.zip?download=1'

os.makedirs(EXTRACT_DIR, exist_ok=True)

print('Working dir free space:')
os.system('df -h /kaggle/working')

Working dir free space:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  128K   20G   1% /kaggle/working


0

In [2]:
# ── Step 1: Download subset0 from Zenodo ──
# -c flag = resume if interrupted
# --progress=bar:force forces a visible progress bar in Kaggle notebooks
# Expect ~15-25 min depending on Kaggle <-> Zenodo connection speed

print('Starting download of subset0.zip (~12GB)...')
print('If this cell seems stuck, check: the progress bar may take 30s to appear.\n')

exit_code = os.system(
    f'wget -c "{ZENODO_URL}" -O "{ZIP_PATH}" --progress=bar:force 2>&1'
)

if exit_code == 0:
    size_gb = os.path.getsize(ZIP_PATH) / (1024**3)
    print(f'\nDownload complete. File size: {size_gb:.2f} GB')
else:
    print(f'\nwget failed with exit code {exit_code}.')
    print('Try the fallback cell below using curl instead.')

Starting download of subset0.zip (~12GB)...
If this cell seems stuck, check: the progress bar may take 30s to appear.

--2026-03-19 19:49:37--  https://zenodo.org/records/3723295/files/subset0.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 137.138.153.219, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6811924508 (6.3G) [application/octet-stream]
Saving to: ‘/kaggle/working/subset0.zip’

/kaggle/working/sub 100%[===================>]   6.34G  21.3MB/s    in 5m 20s  

2026-03-19 19:54:58 (20.3 MB/s) - ‘/kaggle/working/subset0.zip’ saved [6811924508/6811924508]


Download complete. File size: 6.34 GB


In [3]:
# ── Fallback: use curl if wget failed ──
# Only run this cell if the wget cell above failed.

if not os.path.exists(ZIP_PATH) or os.path.getsize(ZIP_PATH) < 1e9:
    print('wget failed or file too small. Trying curl...')
    exit_code = os.system(
        f'curl -L "{ZENODO_URL}" -o "{ZIP_PATH}" --progress-bar'
    )
    if exit_code == 0:
        size_gb = os.path.getsize(ZIP_PATH) / (1024**3)
        print(f'curl complete. File size: {size_gb:.2f} GB')
    else:
        print('curl also failed. Zenodo may be rate-limiting Kaggle IPs.')
        print('In that case, download subset0.zip to your machine and upload manually:')
        print('  kaggle.com → Datasets → New Dataset → upload subset0.zip')
else:
    print(f'subset0.zip already exists ({os.path.getsize(ZIP_PATH)/(1024**3):.2f} GB). Skipping curl.')

subset0.zip already exists (6.34 GB). Skipping curl.


In [4]:
# ── Step 2: Verify the zip is intact before extracting ──

print('Verifying zip integrity...')
result = os.system(f'unzip -t "{ZIP_PATH}" > /dev/null 2>&1')

if result == 0:
    print('Zip is valid. Ready to extract.')
else:
    print('WARNING: Zip appears corrupted or incomplete.')
    print('This usually means the download was interrupted.')
    print('Re-run the wget cell — the -c flag will resume from where it stopped.')

Verifying zip integrity...
Zip is valid. Ready to extract.


In [5]:
# ── Step 3: Extract subset0.zip ──
# LUNA16 zips contain .mhd + .raw file pairs
# .mhd = header (human-readable metadata)
# .raw = the actual voxel data (binary, referenced by the .mhd)
# Both files MUST stay in the same directory — never separate them

print(f'Extracting to {EXTRACT_DIR} ...')
print('Expect ~3-5 min for extraction.\n')

exit_code = os.system(f'unzip -q "{ZIP_PATH}" -d "{EXTRACT_DIR}"')

if exit_code == 0:
    print('Extraction complete.')
else:
    print(f'unzip exited with code {exit_code} — may still be OK if some files were skipped.')

Extracting to /kaggle/working/luna16_subset0 ...
Expect ~3-5 min for extraction.

Extraction complete.


In [6]:
# ── Step 4: Verify extracted contents ──

import glob

mhd_files = sorted(glob.glob(os.path.join(EXTRACT_DIR, '**', '*.mhd'), recursive=True))
raw_files = sorted(glob.glob(os.path.join(EXTRACT_DIR, '**', '*.raw'), recursive=True))

print(f'.mhd files found : {len(mhd_files)}')
print(f'.raw files found : {len(raw_files)}')

if len(mhd_files) != len(raw_files):
    print('WARNING: mhd and raw counts differ — some files may be missing.')
elif len(mhd_files) == 0:
    print('WARNING: No .mhd files found. Check extraction path.')
else:
    print(f'\nLooks good — {len(mhd_files)} CT volumes ready.')
    print('\nFirst 3 files:')
    for f in mhd_files[:3]:
        print(f'  {os.path.basename(f)}')

.mhd files found : 89
.raw files found : 89

Looks good — 89 CT volumes ready.

First 3 files:
  1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031515062000744821260.mhd
  1.3.6.1.4.1.14519.5.2.1.6279.6001.108197895896446896160048741492.mhd
  1.3.6.1.4.1.14519.5.2.1.6279.6001.109002525524522225658609808059.mhd


In [7]:
# ── Step 5: Quick sanity check on one volume ──
# Confirms HU values look correct before you delete the zip

import SimpleITK as sitk
import numpy as np

test_file = mhd_files[0]
print(f'Reading: {os.path.basename(test_file)}')

img   = sitk.ReadImage(test_file)
arr   = sitk.GetArrayFromImage(img)

print(f'  Shape   : {arr.shape}  (Z, Y, X)')
print(f'  dtype   : {arr.dtype}')
print(f'  HU min  : {arr.min()}')
print(f'  HU max  : {arr.max()}')
print(f'  Spacing : {img.GetSpacing()} mm')
print(f'  Origin  : {img.GetOrigin()} mm')

# Sanity: lung HU should be roughly -1350 to +150 after windowing
# Raw CT HU range is typically -1500 to +3000
if arr.min() < -500 and arr.max() > 100:
    print('\nHU range looks correct — this is a raw CT volume.')
else:
    print('\nWARNING: unexpected HU range. This may not be a raw CT.')

Reading: 1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031515062000744821260.mhd
  Shape   : (121, 512, 512)  (Z, Y, X)
  dtype   : int16
  HU min  : -3024
  HU max  : 2103
  Spacing : (0.7617189884185791, 0.7617189884185791, 2.5) mm
  Origin  : (-198.100006, -195.0, -335.209991) mm

HU range looks correct — this is a raw CT volume.


In [8]:
# ── Step 6: Delete the zip to free space ──
# The extracted folder is what we need. The zip just wastes 12GB.

os.remove(ZIP_PATH)
print(f'Deleted {ZIP_PATH}')

print('\nFinal disk usage:')
os.system('df -h /kaggle/working')

print('\nExtracted folder size:')
os.system(f'du -sh "{EXTRACT_DIR}"')

print()
print('='*55)
print('NEXT STEPS:')
print('='*55)
print('1. Click "Save Version" in the top-right of this notebook')
print('2. Choose "Save and Run All" — let it finish')
print('3. After it completes, go to the Output tab')
print('4. Find luna16_subset0/ folder → click the three dots')
print('5. Click "Create Dataset from Output"')
print('6. Name it: luna16-subset0-raw-ct')
print('7. Add that dataset to your preprocessing notebook')
print()
print('The path in your preprocessing notebook will be:')
print('/kaggle/input/luna16-subset0-raw-ct/luna16_subset0/subset0/')
print('(verify the exact subfolder name in the output tab)')

Deleted /kaggle/working/subset0.zip

Final disk usage:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   12G  8.4G  58% /kaggle/working

Extracted folder size:
12G	/kaggle/working/luna16_subset0

NEXT STEPS:
1. Click "Save Version" in the top-right of this notebook
2. Choose "Save and Run All" — let it finish
3. After it completes, go to the Output tab
4. Find luna16_subset0/ folder → click the three dots
5. Click "Create Dataset from Output"
6. Name it: luna16-subset0-raw-ct
7. Add that dataset to your preprocessing notebook

The path in your preprocessing notebook will be:
/kaggle/input/luna16-subset0-raw-ct/luna16_subset0/subset0/
(verify the exact subfolder name in the output tab)
